In [1]:
%uv pip install pandas scikit-learn vitaldb ipywidgets jupyter

Using Python 3.10.13 environment at: /usr/local
Resolved 125 packages in 530ms
⠙ Preparing packages... (0/62)
⠙ Preparing packages... (0/62)
⠙ Preparing packages... (0/62)
⠙ Preparing packages... (0/62)
arrow                ------------------------------ 14.85 KiB/67.18 KiB
⠙ Preparing packages... (0/62)
arrow                ------------------------------ 14.85 KiB/67.18 KiB
⠙ Preparing packages... (0/62)
arrow                ------------------------------ 14.85 KiB/67.18 KiB
⠙ Preparing packages... (0/62)
arrow                ------------------------------ 14.85 KiB/67.18 KiB
⠙ Preparing packages... (0/62)
jupyterlab-pygments  ------------------------------     0 B/15.51 KiB
arrow                ------------------------------ 14.85 KiB/67.18 KiB
⠙ Preparing packages... (0/62)
jupyterlab-pygments  ------------------------------ 14.87 KiB/15.51 KiB
arrow                ------------------------------ 14.85 KiB/67.18 KiB
⠙ Preparing packages... (0/62)
jupyterlab-pygments  ----------------

In [3]:
"""
plot_figures.py
===============
Generates Figures 3, 4, and 5 for the IOH prediction paper.

BEFORE RUNNING:
- Fill in all sections marked with # <<< FILL IN >>>
- Required: numpy, matplotlib, sklearn
- Run from your project directory
"""

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from sklearn.metrics import roc_curve, precision_recall_curve, auc

# ── Global style ──────────────────────────────────────────────────────────────
plt.rcParams.update({
    "font.family":       "serif",
    "font.size":         10,
    "axes.titlesize":    11,
    "axes.labelsize":    10,
    "legend.fontsize":   8.5,
    "xtick.labelsize":   9,
    "ytick.labelsize":   9,
    "figure.dpi":        300,
    "savefig.dpi":       300,
    "savefig.bbox":      "tight",
    "axes.spines.top":   False,
    "axes.spines.right": False,
})

# ── Colours and line styles ───────────────────────────────────────────────────
# Chosen to be distinguishable in greyscale print
STYLES = {
    "M1": {"color": "#000000", "lw": 2.2, "ls": "-",  "label": "Mamba D1 (Full)"},
    "M2": {"color": "#E69F00", "lw": 1.6, "ls": "--", "label": "Mamba D2 (Vitals only)"},
    "M5": {"color": "#56B4E9", "lw": 1.6, "ls": ":",  "label": "Mamba D5 (Drugs only)"},
    "T1": {"color": "#CC79A7", "lw": 1.6, "ls": "-.", "label": "Transformer D1 (Full)"},
}


# ═══════════════════════════════════════════════════════════════════════════════
# FIGURE 3 — ROC and PRC Curves
# ═══════════════════════════════════════════════════════════════════════════════

def plot_figure3():
    """
    ROC and PRC curves for M1, M2, M5, T1.

    WHAT YOU NEED TO FILL IN:
    Each model needs two arrays loaded from your saved predictions:
        y_true  : 1-D array of ground truth labels  (shape: [n_windows])
        y_score : 1-D array of predicted probabilities (shape: [n_windows])

    These should be the pooled predictions across all three seeds for each model.
    Load them from wherever you saved your model outputs (e.g. .npy or .pkl files).

    The dict below maps model ID → (y_true, y_score).
    """

    predictions = {

        # <<< FILL IN >>>
        # Replace None with your actual arrays, e.g.:
        #   np.load("outputs/m1_y_true.npy"), np.load("outputs/m1_y_score.npy")
        "M1": (np.load("/root/m1_y_true.npy"), np.load("/root/m1_y_score.npy")),   # y_true, y_score for Mamba D1

        # <<< FILL IN >>>
        "M2": (np.load("/root/m2_y_true.npy"), np.load("/root/m2_y_score.npy")),   # y_true, y_score for Mamba D2 (vitals only)

        # <<< FILL IN >>>
        "M5": (np.load("/root/m5_y_true.npy"), np.load("/root/m5_y_score.npy")),   # y_true, y_score for Mamba D5 (drugs only)

        # <<< FILL IN >>>
        "T1": (np.load("/root/t1_y_true.npy"), np.load("/root/t1_y_score.npy")),   # y_true, y_score for Transformer D1
    }

    # Known AUROC / AUPRC values (from main_evals.txt) for legend labels
    known_auroc = {"M1": 0.7360, "M2": 0.7069, "M5": 0.6306, "T1": 0.7430}
    known_auprc = {"M1": 0.1794, "M2": 0.1545, "M5": 0.1200, "T1": 0.1824}

    # Random guessing PRC baseline (= positive prevalence in test set)
    RANDOM_AUPRC_BASELINE = 0.0657  # already correct, do not change

    fig, axes = plt.subplots(1, 2, figsize=(10, 4.5))
    ax_roc, ax_prc = axes

    for model_id, (y_true, y_score) in predictions.items():
        s = STYLES[model_id]

        # ── ROC ──
        fpr, tpr, _ = roc_curve(y_true, y_score)
        roc_auc     = auc(fpr, tpr)
        ax_roc.plot(fpr, tpr,
                    color=s["color"], lw=s["lw"], ls=s["ls"],
                    label=f"{s['label']} (AUROC={known_auroc[model_id]:.4f})")

        # ── PRC ──
        precision, recall, _ = precision_recall_curve(y_true, y_score)
        ax_prc.plot(recall, precision,
                    color=s["color"], lw=s["lw"], ls=s["ls"],
                    label=f"{s['label']} (AUPRC={known_auprc[model_id]:.4f})")

    # ── ROC decorations ──
    ax_roc.plot([0, 1], [0, 1], "k--", lw=1.0, label="Random (AUROC=0.50)")
    ax_roc.set_xlabel("False Positive Rate")
    ax_roc.set_ylabel("True Positive Rate")
    ax_roc.set_title("ROC Curves")
    ax_roc.set_xlim([0, 1])
    ax_roc.set_ylim([0, 1])
    ax_roc.legend(loc="lower right")

    # ── PRC decorations ──
    ax_prc.axhline(y=RANDOM_AUPRC_BASELINE, color="k", ls="--", lw=1.0,
                   label=f"Random baseline (AUPRC={RANDOM_AUPRC_BASELINE})")
    ax_prc.set_xlabel("Recall")
    ax_prc.set_ylabel("Precision")
    ax_prc.set_title("Precision-Recall Curves")
    ax_prc.set_xlim([0, 1])
    ax_prc.set_ylim([0, 1])
    ax_prc.legend(loc="upper right")

    plt.tight_layout()
    plt.savefig("figure3_roc_prc.pdf")
    plt.savefig("figure3_roc_prc.png")
    print("Saved figure3_roc_prc.pdf / .png")
    plt.close()


# ═══════════════════════════════════════════════════════════════════════════════
# FIGURE 4 — Lead Time Histogram (M1 only)
# ═══════════════════════════════════════════════════════════════════════════════

def plot_figure4():
    """
    Histogram of true positive lead times in minutes for M1.

    WHAT YOU NEED TO FILL IN:
    lead_times_minutes : 1-D array of lead times (in minutes) for every true
                         positive prediction made by M1 across the test set.

    Lead time = (pred_start_sample - obs_end_sample) / 30 samples-per-minute
              + position of IOH onset within prediction window.

    This should be computable from your saved M1 test predictions and the
    window metadata (start indices, IOH onset indices).

    The values will fall between 5.0 and 15.0 minutes by design.
    Your summary stats: mean=10.0 min, std=2.9 min (from main_evals.txt)
    """

    # <<< FILL IN >>>
    # Replace with your actual lead time array, e.g.:
    #   lead_times_minutes = np.load("outputs/m1_lead_times.npy")
    lead_times_minutes = np.load("/root/m1_lead_times_minutes.npy")

    # Summary stats — already known, do not change

    MEAN_LEAD_TIME = float(np.mean(lead_times_minutes))
    STD_LEAD_TIME  = float(np.std(lead_times_minutes))

    # Bin edges: 5 to 15 minutes in 1-minute bins (11 edges = 10 bins)
    bins = np.arange(5.0, 16.0, 1.0)

    fig, ax = plt.subplots(figsize=(6, 4))

    ax.hist(lead_times_minutes, bins=bins,
            color="#4477AA", edgecolor="white", linewidth=0.6, alpha=0.85)

    # Mean line
    ax.axvline(x=MEAN_LEAD_TIME, color="#BB5566", lw=1.8, ls="--",
               label=f"Mean = {MEAN_LEAD_TIME:.4f} ± {STD_LEAD_TIME:.4f} min")

    # Shaded ±1 std band
    ax.axvspan(MEAN_LEAD_TIME - STD_LEAD_TIME,
               MEAN_LEAD_TIME + STD_LEAD_TIME,
               alpha=0.12, color="#BB5566", label="±1 SD")

    ax.set_xlabel("Lead Time Before IOH Onset (minutes)")
    ax.set_ylabel("True Positive Predictions (count)")
    ax.set_title("Distribution of True Positive Lead Times — Mamba D1 (M1)")
    ax.set_xlim([5, 15])
    ax.set_xticks(np.arange(5, 16, 1))
    ax.legend()

    # Caption note — placed as a figure text, not in the plot itself.
    # Use this verbatim in your LaTeX \caption{}:
    # "Distribution of lead times for true positive IOH predictions by M1.
    #  Bounds of 5–15 minutes are determined by the windowing structure
    #  (5-minute lead gap + 10-minute prediction window), not model behaviour."

    plt.tight_layout()
    plt.savefig("figure4_lead_time.pdf")
    plt.savefig("figure4_lead_time.png")
    print("Saved figure4_lead_time.pdf / .png")
    plt.close()


# ═══════════════════════════════════════════════════════════════════════════════
# FIGURE 5 — VRAM Scaling Curve
# ═══════════════════════════════════════════════════════════════════════════════

def plot_figure5():
    """
    Peak VRAM vs sequence length for Mamba (recurrent) and Transformer.
    All numbers already confirmed from your benchmark — no fill-in needed here.
    """

    # ── Data (confirmed from your benchmark runs) ─────────────────────────────
    seq_lengths = [900, 1500, 2000, 2500, 3000]

    vram_transformer = [36.42, 65.35, 99.04, 142.27, 195.70]  # MB
    vram_mamba       = [9.34,   9.54,  9.54,   9.54,   9.54]  # MB — flat

    # ── Plot ──────────────────────────────────────────────────────────────────
    fig, ax = plt.subplots(figsize=(7, 4.5))

    ax.plot(seq_lengths, vram_transformer,
            color="#CC79A7", lw=2.0, ls="-", marker="o", markersize=5,
            label="Transformer")

    ax.plot(seq_lengths, vram_mamba,
            color="#000000", lw=2.0, ls="-", marker="o", markersize=5,
            label="Mamba (recurrent)")

    # Operational sequence length marker
    ax.axvline(x=900, color="#888888", lw=1.2, ls="--")
    ax.text(920, 185, "Operational\nsequence length\n(900 timesteps)",
            fontsize=8, color="#555555", va="top")

    # Annotate Mamba flat line
    ax.annotate("O(1) — constant memory\n(9.54 MB)",
                xy=(2500, 9.54), xytext=(1600, 55),
                fontsize=8, color="#000000",
                arrowprops=dict(arrowstyle="->", color="#000000", lw=1.0))

    # Annotate Transformer endpoint
    ax.annotate("195.70 MB",
            xy=(3000, 195.70), xytext=(2750, 210),
            fontsize=8, color="#CC79A7",
            arrowprops=dict(arrowstyle="->", color="#CC79A7", lw=1.0))

    ax.set_xlabel("Sequence Length (timesteps)")
    ax.set_ylabel("Peak VRAM (MB)")
    ax.set_title("Peak VRAM vs Sequence Length — Mamba vs Transformer")
    ax.set_xlim([700, 3200])
    ax.set_ylim([0, 230])
    ax.set_xticks(seq_lengths)
    ax.legend(loc="upper left")

    plt.tight_layout()
    plt.savefig("figure5_vram_scaling.pdf")
    plt.savefig("figure5_vram_scaling.png")
    print("Saved figure5_vram_scaling.pdf / .png")
    plt.close()


# ═══════════════════════════════════════════════════════════════════════════════
# ENTRY POINT
# ═══════════════════════════════════════════════════════════════════════════════

if __name__ == "__main__":
    # Comment out any figure you're not ready to generate yet
    # (Figure 3 and 4 require fill-in data before they will run)

    # plot_figure3()   # <<< Needs predictions arrays filled in first
    # plot_figure4()   # <<< Needs lead_times_minutes array filled in first
    plot_figure5()     # Ready to run immediately — all data confirmed

Saved figure5_vram_scaling.pdf / .png
